# 🚀 MPLADS AI Risk Intelligence: Advanced Model Training & Accuracy Optimization
**Smart India Hackathon 2026 — Problem Statement:** `SIH26102`  
**Ministry:** MoSPI / Data Informatics & Innovation Division (DIID)  

### Objectives of this Notebook:
1. **Feature Engineering**: Compute 51 temporal, financial ratio, and peer-group features.
2. **Hyperparameter Tuning & Ensemble Optimization**: Train and optimize **XGBoost**, **LightGBM**, and **Random Forest** models to maximize **F1-Score, Precision, and ROC-AUC (>85%+)**.
3. **Unsupervised Anomaly Modeling**: Peer-group calibrated Isolation Forest.
4. **Semantic Duplicate Work Finder**: NLP Vector embeddings + Haversine distance.
5. **Multi-Signal Risk Fusion**: Output final calibrated 0–100 composite risk scores.

## 1. Feature Engineering & Dataset Splitting

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)
import xgboost as xgb

from src.features.build_features import build_all_features

master_path = Path("../data/processed/master_works.parquet")
features_path = Path("../data/processed/features_master.parquet")

df_feat = build_all_features(master_path, features_path)
print(f"✅ Feature Matrix Built: {df_feat.shape[0]} works, {df_feat.shape[1]} engineered features.")

## 2. Advanced Supervised Model Tuning (Optimizing Delay & Stall Prediction)

In [2]:
num_cols = [
    "sanctioned_amount", "rec_to_sanction_days", "planned_duration_days",
    "cost_vs_peer_median_ratio", "peer_cost_zscore", "agency_avg_delay",
    "agency_completion_rate", "district_completion_rate", "schedule_overrun_ratio"
]
cat_cols = ["sector", "state"]

X = df_feat[num_cols + cat_cols].copy()
y = df_feat["target_is_delayed"].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

# Train Tuned XGBoost Classifier
tuned_xgb = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    random_state=42,
    eval_metric="logloss"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", tuned_xgb)
])

pipeline.fit(X_train, y_train)
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print("🎯 OPTIMIZED MODEL PERFORMANCE ON TEST SET:")
print(f"  • Accuracy:        {acc*100:.2f}%")
print(f"  • F1-Score:        {f1*100:.2f}%")
print(f"  • ROC-AUC Score:   {roc:.4f}")
print(f"  • Precision:       {prec*100:.2f}%")
print(f"  • Recall:          {rec*100:.2f}%")

## 3. Evaluation Curves: ROC & Precision-Recall Visualization

In [3]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'Tuned XGBoost (ROC-AUC = {roc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].legend(loc='lower right')

# PR Curve
axes[1].plot(recall_curve, precision_curve, color='#10b981', lw=2.5, label=f'Precision-Recall Curve (F1 = {f1*100:.1f}%)')
axes[1].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

## 4. Save Optimized Model Artifacts

In [4]:
# Fit on full dataset and save
pipeline.fit(X, y)
model_save_path = Path("../models/delay_model.joblib")
joblib.dump(pipeline, model_save_path)
print(f"💾 Successfully saved optimized delay prediction model to {model_save_path}")